In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
%pip install faker

# catalog = "wk_dev_edu"  # Substitua pelo nome do seu catálogo
schema = "bronze"


from pyspark.sql import functions as F
from faker import Faker
import uuid
import random

fake = Faker()

fake_firstname = F.udf(fake.first_name)
fake_lastname = F.udf(fake.last_name)
fake_email = F.udf(fake.ascii_company_email)
fake_date = F.udf(lambda: fake.date_time_this_month().strftime("%Y-%m-%d %H:%M:%S"))
fake_address = F.udf(fake.address)
fake_id = F.udf(lambda: str(uuid.uuid4()) if random.uniform(0, 1) < 0.98 else None)

df = spark.range(0, 1000)
df = df.withColumn("id", fake_id())
df = df.withColumn("firstname", fake_firstname())
df = df.withColumn("lastname", fake_lastname())
df = df.withColumn("email", fake_email())
df = df.withColumn("address", fake_address())
df = df.withColumn("created_at", fake_date())

df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.clientes_ficticios")

